# 04 — Base Detection

## Goal
Identify **base clusters** — consecutive candles where price is consolidating rather than trending.  
A base is the "loading zone" between two legs; it's where institutional orders are placed.

## What Makes a Candle a "Base" Candle?
Two conditions must both be true:

$$\text{body\_ratio} = \frac{|\text{close} - \text{open}|}{\text{high} - \text{low}} \leq 0.50$$

$$\frac{\text{candle\_range}}{\text{ATR}} \leq 1.5$$

The first condition rejects candles with a large body (strong direction).  
The second condition rejects candles with a large range even if their body is tiny — e.g. a doji on a volatile 5-point swing still isn't a calm base candle. *(This was the Scenario D bug.)*

## Cluster Filters
Once a cluster of base candles is found, it must pass **three tightness gates**:

| Filter | Equation | Threshold |
|--------|----------|-----------|
| Min length | $\text{count} \geq 2$ | At least 2 candles |
| Max length | $\text{count} \leq 5$ | No more than 5 candles |
| Width / ATR | $\dfrac{\text{base\_high} - \text{base\_low}}{\text{avg\_ATR}} \leq 2.5$ | Cluster is tight |
| Compactness | $\dfrac{\text{width / ATR}}{\text{count}} \leq 0.80$ | Tight relative to its length |

## The Bug We Found (Scenario D)
Scenario D has a wide candle with a tiny body (`body_ratio = 0.145`) but a huge range (5.5 pts, ATR ≈ 1.8).  
`range / ATR = 3.05` — this candle is **not calm**, it's just a doji on a big swing.  
The original `is_base` check (body-ratio only) incorrectly marked it `True`.  
Adding the `range / ATR ≤ 1.5` gate fixes it.

## Visualization
The chart highlights all detected base clusters as grey shaded boxes overlaid on the candlestick chart.


In [7]:
import pandas as pd

BASE_BODY_RATIO_MAX: float = 0.50   # candle is "base" if body / range <= 0.50
BASE_MIN_CANDLES: int = 1           # cluster must have at least 1 base candle
BASE_MAX_CANDLES: int = 5           # cluster cannot exceed 5 candles
BASE_MAX_ATR_WIDTH: float = 2.5     # (base_high - base_low) / ATR  <=  2.5
BASE_COMPACTNESS_MAX: float = 0.80  # tightness vs candle count

df = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
df.head()


,open,high,low,close,volume,range,body,body-ratio,is_base,is_doji,is_bullish,atr
Date,,,,,,,,,,,,
2024-01-01 00:00:00+00:00,100.0,101.1,99.5,100.6,1000000,1.6,0.6,0.375000,True,False,True,1.600000
2024-01-08 00:00:00+00:00,100.6,101.1,99.6,100.1,1000000,1.5,0.5,0.333333,True,False,False,1.592857
2024-01-15 00:00:00+00:00,100.1,101.2,99.6,100.7,1000000,1.6,0.6,0.375000,True,False,True,1.593367
2024-01-22 00:00:00+00:00,100.7,101.2,99.7,100.2,1000000,1.5,0.5,0.333333,True,False,False,1.586698
2024-01-29 00:00:00+00:00,100.2,101.3,99.7,100.8,1000000,1.6,0.6,0.375000,True,False,True,1.587648


In [8]:
def find_base_clusters(df: pd.DataFrame) -> list[dict]:
    """
    Mimics the first half of detect_zones() in zone_detector.py:
    walks candles, starts a cluster on a base candle, extends while still base
    and within BASE_MAX_CANDLES.

    Pipeline contract: assumes `is_base` was added by the enrichment step.
    """
    required = {"is_base"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame missing required columns: {missing}. Run enrichment first.")

    clusters = []
    is_base = df["is_base"].to_numpy()
    i = 0
    n = len(df)
    while i < n:
        # Step 1: must START on a base candle
        if not is_base[i]:
            i += 1
            continue

        # Step 2: extend the cluster while next candle is still base
        # and we haven't hit BASE_MAX_CANDLES
        start = i
        end = i
        while (
            end + 1 < n
            and is_base[end + 1]
            and (end - start + 1) < BASE_MAX_CANDLES
        ):
            end += 1

        clusters.append({
            "start": start,
            "end": end,
            "count": end - start + 1,
        })
        i = end + 1
    return clusters


clusters = find_base_clusters(df)
print(f"Found {len(clusters)} base cluster(s)")
for c in clusters[:15]:
    print(c)


Found 27 base cluster(s)
{'start': 0, 'end': 4, 'count': 5}
{'start': 5, 'end': 9, 'count': 5}
{'start': 10, 'end': 14, 'count': 5}
{'start': 15, 'end': 16, 'count': 2}
{'start': 19, 'end': 21, 'count': 3}
{'start': 23, 'end': 23, 'count': 1}
{'start': 27, 'end': 29, 'count': 3}
{'start': 31, 'end': 33, 'count': 3}
{'start': 35, 'end': 37, 'count': 3}
{'start': 39, 'end': 41, 'count': 3}
{'start': 43, 'end': 43, 'count': 1}
{'start': 46, 'end': 47, 'count': 2}
{'start': 50, 'end': 51, 'count': 2}
{'start': 53, 'end': 54, 'count': 2}
{'start': 57, 'end': 58, 'count': 2}


In [9]:
def evaluate_cluster(df: pd.DataFrame, cluster: dict) -> dict:
    s, e = cluster["start"], cluster["end"]
    sub = df.iloc[s:e + 1]
    base_high = sub["high"].max()
    base_low = sub["low"].min()
    base_width = base_high - base_low
    avg_atr = sub["atr"].mean()
    width_atr_ratio = base_width / avg_atr
    compactness_ratio = width_atr_ratio / cluster["count"]
    return {
        **cluster,
        "base_high": round(base_high, 3),
        "base_low": round(base_low, 3),
        "base_width": round(base_width, 3),
        "avg_atr": round(avg_atr, 3),
        "width_atr_ratio": round(width_atr_ratio, 3),
        "compactness_ratio": round(compactness_ratio, 3),
        "width_passed": width_atr_ratio <= BASE_MAX_ATR_WIDTH,
        "compactness_passed": compactness_ratio <= BASE_COMPACTNESS_MAX,
        "min_count_passed": cluster["count"] >= BASE_MIN_CANDLES,
    }


evaluated = [evaluate_cluster(df, c) for c in clusters]
pd.DataFrame(evaluated)


,start,end,count,base_high,base_low,base_width,avg_atr,width_atr_ratio,compactness_ratio,width_passed,compactness_passed,min_count_passed
0,0,4,5,101.3,99.5,1.8,1.592,1.131,0.226,True,True,True
1,5,9,5,101.5,99.8,1.7,1.578,1.077,0.215,True,True,True
2,10,14,5,101.8,100.0,1.8,1.570,1.146,0.229,True,True,True
3,15,16,2,101.8,99.6,2.2,1.551,1.418,0.709,True,True,True
4,19,21,3,105.9,104.9,1.0,1.587,0.630,0.210,True,True,True
5,23,23,1,110.4,109.0,1.4,1.744,0.803,0.803,True,False,True
6,27,29,3,106.7,105.8,0.9,1.756,0.513,0.171,True,True,True
7,31,33,3,102.3,100.2,2.1,1.801,1.166,0.389,True,True,True
8,35,37,3,104.0,103.0,1.0,1.677,0.596,0.199,True,True,True
9,39,41,3,104.4,103.2,1.2,1.440,0.834,0.278,True,True,True


In [10]:
# Find the cluster that overlaps Scenario A's base candles.
# Scenario A base candles in fixtures_labeled.csv have close ~ 105.3 / 105.5 / 105.2
scenario_a_mask = df["close"].between(105.0, 106.0) & df["is_base"]
scenario_a_positions = [df.index.get_loc(idx) for idx in df[scenario_a_mask].index]
print("Candidate base candle positions for Scenario A:", scenario_a_positions)

cluster_a = next(c for c in clusters if c["start"] in scenario_a_positions)
print("\nDetected cluster A:", cluster_a)

result_a = evaluate_cluster(df, cluster_a)
print("\nEvaluation:")
for k, v in result_a.items():
    print(f"  {k}: {v}")

print("\nRaw base candles:")
df.iloc[cluster_a["start"]:cluster_a["end"] + 1][["open", "high", "low", "close", "body-ratio", "is_base", "atr"]]


Candidate base candle positions for Scenario A: [19, 20, 21, 29, 43]

Detected cluster A: {'start': 19, 'end': 21, 'count': 3}

Evaluation:
  start: 19
  end: 21
  count: 3
  base_high: 105.9
  base_low: 104.9
  base_width: 1.0
  avg_atr: 1.587
  width_atr_ratio: 0.63
  compactness_ratio: 0.21
  width_passed: True
  compactness_passed: True
  min_count_passed: True

Raw base candles:


,open,high,low,close,body-ratio,is_base,atr
Date,,,,,,,
2024-05-13 00:00:00+00:00,105.4,105.8,105.0,105.3,0.125,True,1.646234
2024-05-20 00:00:00+00:00,105.3,105.7,104.9,105.5,0.250,True,1.585789
2024-05-27 00:00:00+00:00,105.5,105.9,105.1,105.2,0.375,True,1.529661


In [11]:
# Locate Scenario D's two "WIDE bar" candles by exact open/close signature
mask_d = (
    ((df["open"].round(2) == 106.80) & (df["close"].round(2) == 106.00)) |
    ((df["open"].round(2) == 106.00) & (df["close"].round(2) == 110.50))
)
wide_rows = df[mask_d][["open", "high", "low", "close", "range", "body", "body-ratio", "is_base", "atr"]]
print("Scenario D 'wide' candles:")
print(wide_rows)

wide_positions = [df.index.get_loc(idx) for idx in wide_rows.index]
print("\nPositions in df:", wide_positions)

# Did our loop pick them up as a cluster?
matching = [c for c in clusters if c["start"] in wide_positions or c["end"] in wide_positions]
print("\nClusters overlapping these positions:", matching)

# Force-evaluate them AS IF they formed a cluster, to prove the tightness filter
# WOULD reject them if they ever reached the filter.
forced_cluster = {"start": wide_positions[0], "end": wide_positions[-1],
                  "count": wide_positions[-1] - wide_positions[0] + 1}
print("\nForced evaluation (bypassing body-ratio gate):")
forced = evaluate_cluster(df, forced_cluster)
for k, v in forced.items():
    print(f"  {k}: {v}")


Scenario D 'wide' candles:
                            open   high    low  close  range  body  \
Date                                                                 
2024-10-28 00:00:00+00:00  106.8  110.5  105.0  106.0    5.5   0.8   
2024-11-04 00:00:00+00:00  106.0  111.0  104.5  110.5    6.5   4.5   

                           body-ratio  is_base       atr  
Date                                                      
2024-10-28 00:00:00+00:00    0.145455     True  1.805965  
2024-11-04 00:00:00+00:00    0.692308    False  2.141253  

Positions in df: [43, 44]

Clusters overlapping these positions: [{'start': 43, 'end': 43, 'count': 1}]

Forced evaluation (bypassing body-ratio gate):
  start: 43
  end: 44
  count: 2
  base_high: 111.0
  base_low: 104.5
  base_width: 6.5
  avg_atr: 1.974
  width_atr_ratio: 3.293
  compactness_ratio: 1.647
  width_passed: False
  compactness_passed: False
  min_count_passed: True


In [12]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

COLOR_BULL  = "#26a69a"
COLOR_BEAR  = "#ef5350"
COLOR_BASE  = "#b0bec5"
BG          = "#131722"
GRID        = "#1e222d"

# ── evaluate all clusters and split into passed / failed ───────────────────
passed = [e for e in evaluated if e["width_passed"] and e["compactness_passed"] and e["min_count_passed"]]
failed = [e for e in evaluated if not (e["width_passed"] and e["compactness_passed"] and e["min_count_passed"])]

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.75, 0.25],
    vertical_spacing=0.03,
)

# ── candles ────────────────────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"],
    low=df["low"],   close=df["close"],
    increasing_line_color=COLOR_BULL,
    decreasing_line_color=COLOR_BEAR,
    name="Price",
), row=1, col=1)

# ── base cluster rectangles ────────────────────────────────────────────────
for cluster in passed:
    s, e = cluster["start"], cluster["end"]
    x0, x1 = df.index[s], df.index[e]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor="rgba(38,166,154,0.15)",
        line=dict(color="#26a69a", width=1, dash="dot"),
        row=1, col=1,
        annotation_text=f"✓ {cluster['count']}c",
        annotation_font=dict(size=9, color="#26a69a"),
        annotation_position="top left",
    )

for cluster in failed:
    s, e = cluster["start"], cluster["end"]
    x0, x1 = df.index[s], df.index[e]
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor="rgba(239,83,80,0.08)",
        line=dict(color="#ef5350", width=1, dash="dot"),
        row=1, col=1,
    )

# ── ATR line ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["atr"],
    mode="lines",
    line=dict(color="#7c83fd", width=1.5),
    fill="tozeroy",
    fillcolor="rgba(124,131,253,0.10)",
    name="ATR(14)",
    hovertemplate="%{x}<br>ATR: %{y:.3f}<extra></extra>",
), row=2, col=1)

# ── layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title=f"Base Detection — {len(passed)} valid cluster(s)  |  {len(failed)} rejected",
    height=620,
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color="white",
    xaxis_rangeslider_visible=False,
    hovermode="x unified",
    legend=dict(orientation="h", y=1.04),
)

shared_axis = dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)
fig.update_xaxes(**shared_axis)
fig.update_yaxes(**shared_axis)
fig.update_yaxes(title_text="Price",  row=1, col=1)
fig.update_yaxes(title_text="ATR(14)", row=2, col=1)
fig.update_xaxes(showticklabels=True, row=2, col=1)

fig.show()
